In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import os
print(os.listdir("/content/drive/MyDrive/Cataract/Data/Train"))

['Cataract', 'Normal', 'Not Eye']


In [3]:
import IPython
display(IPython.display.Javascript('''
function ClickConnect(){
    console.log("Keeping alive...");
    var buttons = document.querySelectorAll("colab-toolbar-button");
    for(var i=0;i<buttons.length;i++){
        if(buttons[i].id=="connect"){buttons[i].click();}
    }
}
setInterval(ClickConnect, 60000)
'''))
print("Keep-alive started")

<IPython.core.display.Javascript object>

Keep-alive started


In [4]:
import numpy as np
import pandas as pd
import os
import time
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras.layers import *
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import InceptionResNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint

from sklearn.metrics import classification_report, confusion_matrix, recall_score, fbeta_score
print("All imports done")

All imports done


In [5]:
TRAIN_PATH = "/content/drive/MyDrive/Cataract/Data/Train"
TEST_PATH  = "/content/drive/MyDrive/Cataract/Data/Test"
MODEL_PATH = "/content/drive/MyDrive/Cataract/InceptionResNetV2_FineTune.h5"

print("Train:", TRAIN_PATH)
print("Test: ", TEST_PATH)
print("Model:", MODEL_PATH)

Train: /content/drive/MyDrive/Cataract/Data/Train
Test:  /content/drive/MyDrive/Cataract/Data/Test
Model: /content/drive/MyDrive/Cataract/InceptionResNetV2_FineTune.h5


In [6]:
print("Train folders:", os.listdir(TRAIN_PATH))
print("Test folders: ", os.listdir(TEST_PATH))

Train folders: ['Cataract', 'Normal', 'Not Eye']
Test folders:  ['Cataract', 'Normal', 'Not Eye']


In [7]:
datagen = ImageDataGenerator(
    rescale          = 1./255,
    validation_split = 0.2,
    horizontal_flip  = True,
    vertical_flip    = True
)
print("Datagen ready")

Datagen ready


In [8]:
train_it = datagen.flow_from_directory(
    TRAIN_PATH,
    target_size = (224, 224),
    color_mode  = 'rgb',
    class_mode  = 'categorical',
    batch_size  = 32,
    subset      = "training"
)

val_it = datagen.flow_from_directory(
    TRAIN_PATH,
    target_size = (224, 224),
    color_mode  = 'rgb',
    class_mode  = 'categorical',
    batch_size  = 32,
    subset      = "validation"
)

test_it = datagen.flow_from_directory(
    TEST_PATH,
    target_size = (224, 224),
    color_mode  = 'rgb',
    class_mode  = 'categorical',
    batch_size  = 32,
    shuffle     = False
)

print("Class indices:", train_it.class_indices)

Found 8896 images belonging to 3 classes.
Found 2221 images belonging to 3 classes.
Found 2552 images belonging to 3 classes.
Class indices: {'Cataract': 0, 'Normal': 1, 'Not Eye': 2}


In [9]:
base_model = InceptionResNetV2(
    weights     = 'imagenet',
    input_shape = (224, 224, 3),
    include_top = False
)
print("InceptionResNetV2 base loaded")

219055592/219055592 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
InceptionResNetV2 base loaded


In [10]:
ocl1 = Conv2D(32, (3,3), activation='relu', padding='same')(base_model.output)
bn1  = BatchNormalization()(ocl1)
mp1  = MaxPooling2D(pool_size=(2,2), padding='same')(bn1)
do1  = Dropout(0.17)(mp1)

# ← removed ocl2 Conv2D(64) because feature map becomes too small

al1  = GlobalAveragePooling2D()(do1)

fc1  = Dense(64, activation='relu')(al1)
fc2  = Dense(32, activation='relu')(fc1)
fc3  = Dense(32, activation='relu')(fc2)

al2  = BatchNormalization()(fc3)
all2 = Dropout(0.3)(al2)

output = Dense(3, activation='softmax', name='preds')(all2)
print("CNN head built")

CNN head built


In [11]:
Cataract_Model = Model(inputs=base_model.input, outputs=output)
Cataract_Model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 111, 111,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 111, 111,  │         96 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 111, 111,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 109, 109,  │      9,216 │ activation[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 109, 109,  │         96 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 109, 109,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 109, 109,  │     18,432 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 109, 109,  │        192 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 109, 109,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 54, 54,    │          0 │ activation_2[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 54, 54,    │      5,120 │ max_pooling2d[0]… │
│                     │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 54, 54,    │        240 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 54, 54,    │          0 │ batch_normalizat… │
│ (Activation)        │ 80)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 52, 52,    │    138,240 │ activation_3[0][… │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 52, 52,    │        576 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 52, 52,    │          0 │ batch_normalizat

 Total params: 54,784,739 (208.99 MB)

 Trainable params: 54,724,067 (208.76 MB)

 Non-trainable params: 60,672 (237.00 KB)

In [12]:
Cataract_Model.compile(
    optimizer = Adam(learning_rate=0.0001),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)
print("Model compiled")

Model compiled


In [ ]:
for ix in range(780):
    Cataract_Model.layers[ix].trainable = False

print("First 780 layers frozen")

First 780 layers frozen


In [ ]:
for i, layer in enumerate(Cataract_Model.layers):
    print(i, layer.name, layer.trainable)

0 input_layer False
1 conv2d False
2 batch_normalization False
3 activation False
4 conv2d_1 False
5 batch_normalization_1 False
6 activation_1 False
7 conv2d_2 False
8 batch_normalization_2 False
9 activation_2 False
10 max_pooling2d False
11 conv2d_3 False
12 batch_normalization_3 False
13 activation_3 False
14 conv2d_4 False
15 batch_normalization_4 False
16 activation_4 False
17 max_pooling2d_1 False
18 conv2d_8 False
19 batch_normalization_8 False
20 activation_8 False
21 conv2d_6 False
22 conv2d_9 False
23 batch_normalization_6 False
24 batch_normalization_9 False
25 activation_6 False
26 activation_9 False
27 average_pooling2d False
28 conv2d_5 False
29 conv2d_7 False
30 conv2d_10 False
31 conv2d_11 False
32 batch_normalization_5 False
33 batch_normalization_7 False
34 batch_normalization_10 False
35 batch_normalization_11 False
36 activation_5 False
37 activation_7 False
38 activation_10 False
39 activation_11 False
40 mixed_5b False
41 conv2d_15 False
42 batch_normalization_15

In [ ]:
mc = ModelCheckpoint(
    MODEL_PATH,
    monitor        = 'val_accuracy',
    mode           = 'max',
    verbose        = 1,
    save_best_only = True
)

history = Cataract_Model.fit(
    train_it,
    epochs          = 20,
    validation_data = val_it,
    callbacks       = [mc]
)
print("Training done")

# Save the final in-memory model immediately after training
# This saves in the same session — no CustomScaleLayer issue
Cataract_Model.save(MODEL_PATH)
print("Model saved to:", MODEL_PATH)

Epoch 1/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.7989 - loss: 0.4966
Epoch 1: val_accuracy improved from None to 0.95768, saving model to /content/drive/MyDrive/Cataract/InceptionResNetV2_FineTune.h5



Epoch 1: finished saving model to /content/drive/MyDrive/Cataract/InceptionResNetV2_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 2762s 10s/step - accuracy: 0.8877 - loss: 0.3032 - val_accuracy: 0.9577 - val_loss: 0.1251
Epoch 2/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step - accuracy: 0.9581 - loss: 0.1329
Epoch 2: val_accuracy improved from 0.95768 to 0.97659, saving model to /content/drive/MyDrive/Cataract/InceptionResNetV2_FineTune.h5



Epoch 2: finished saving model to /content/drive/MyDrive/Cataract/InceptionResNetV2_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 76s 274ms/step - accuracy: 0.9563 - loss: 0.1364 - val_accuracy: 0.9766 - val_loss: 0.0864
Epoch 3/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step - accuracy: 0.9689 - loss: 0.1061
Epoch 3: val_accuracy improved from 0.97659 to 0.97884, saving model to /content/drive/MyDrive/Cataract/InceptionResNetV2_FineTune.h5



Epoch 3: finished saving model to /content/drive/MyDrive/Cataract/InceptionResNetV2_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 77s 276ms/step - accuracy: 0.9714 - loss: 0.0998 - val_accuracy: 0.9788 - val_loss: 0.0588
Epoch 4/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step - accuracy: 0.9767 - loss: 0.0824
Epoch 4: val_accuracy did not improve from 0.97884
278/278 ━━━━━━━━━━━━━━━━━━━━ 70s 251ms/step - accuracy: 0.9771 - loss: 0.0821 - val_accuracy: 0.9734 - val_loss: 0.0745
Epoch 5/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 200ms/step - accuracy: 0.9806 - loss: 0.0684
Epoch 5: val_accuracy improved from 0.97884 to 0.98109, saving model to /content/drive/MyDrive/Cataract/InceptionResNetV2_FineTune.h5



Epoch 5: finished saving model to /content/drive/MyDrive/Cataract/InceptionResNetV2_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 75s 268ms/step - accuracy: 0.9807 - loss: 0.0685 - val_accuracy: 0.9811 - val_loss: 0.0592
Epoch 6/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step - accuracy: 0.9835 - loss: 0.0632
Epoch 6: val_accuracy improved from 0.98109 to 0.98919, saving model to /content/drive/MyDrive/Cataract/InceptionResNetV2_FineTune.h5



Epoch 6: finished saving model to /content/drive/MyDrive/Cataract/InceptionResNetV2_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 77s 277ms/step - accuracy: 0.9843 - loss: 0.0608 - val_accuracy: 0.9892 - val_loss: 0.0328
Epoch 7/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step - accuracy: 0.9823 - loss: 0.0641
Epoch 7: val_accuracy did not improve from 0.98919
278/278 ━━━━━━━━━━━━━━━━━━━━ 71s 253ms/step - accuracy: 0.9838 - loss: 0.0566 - val_accuracy: 0.9892 - val_loss: 0.0382
Epoch 8/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step - accuracy: 0.9870 - loss: 0.0441
Epoch 8: val_accuracy did not improve from 0.98919
278/278 ━━━━━━━━━━━━━━━━━━━━ 69s 247ms/step - accuracy: 0.9867 - loss: 0.0443 - val_accuracy: 0.9829 - val_loss: 0.0452
Epoch 9/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step - accuracy: 0.9889 - loss: 0.0419
Epoch 9: val_accuracy did not improve from 0.98919
278/278 ━━━━━━━━━━━━━━━━━━━━ 69s 249ms/step - accuracy: 0.9871 - loss: 0.0454 - val_accuracy: 0.9878 - val_loss: 0.0379
Epoch 


Epoch 13: finished saving model to /content/drive/MyDrive/Cataract/InceptionResNetV2_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 76s 274ms/step - accuracy: 0.9893 - loss: 0.0341 - val_accuracy: 0.9919 - val_loss: 0.0211
Epoch 14/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - accuracy: 0.9881 - loss: 0.0315
Epoch 14: val_accuracy did not improve from 0.99190
278/278 ━━━━━━━━━━━━━━━━━━━━ 70s 251ms/step - accuracy: 0.9900 - loss: 0.0307 - val_accuracy: 0.9905 - val_loss: 0.0244
Epoch 15/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step - accuracy: 0.9926 - loss: 0.0258
Epoch 15: val_accuracy did not improve from 0.99190
278/278 ━━━━━━━━━━━━━━━━━━━━ 68s 245ms/step - accuracy: 0.9918 - loss: 0.0273 - val_accuracy: 0.9901 - val_loss: 0.0265
Epoch 16/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step - accuracy: 0.9920 - loss: 0.0281
Epoch 16: val_accuracy did not improve from 0.99190
278/278 ━━━━━━━━━━━━━━━━━━━━ 69s 249ms/step - accuracy: 0.9927 - loss: 0.0277 - val_accuracy: 0.9874 - val_loss: 0.0365


Epoch 17: finished saving model to /content/drive/MyDrive/Cataract/InceptionResNetV2_FineTune.h5
278/278 ━━━━━━━━━━━━━━━━━━━━ 75s 269ms/step - accuracy: 0.9936 - loss: 0.0247 - val_accuracy: 0.9950 - val_loss: 0.0171
Epoch 18/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 205ms/step - accuracy: 0.9922 - loss: 0.0247
Epoch 18: val_accuracy did not improve from 0.99505
278/278 ━━━━━━━━━━━━━━━━━━━━ 71s 255ms/step - accuracy: 0.9918 - loss: 0.0271 - val_accuracy: 0.9919 - val_loss: 0.0231
Epoch 19/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step - accuracy: 0.9898 - loss: 0.0294
Epoch 19: val_accuracy did not improve from 0.99505
278/278 ━━━━━━━━━━━━━━━━━━━━ 69s 246ms/step - accuracy: 0.9926 - loss: 0.0226 - val_accuracy: 0.9950 - val_loss: 0.0171
Epoch 20/20
278/278 ━━━━━━━━━━━━━━━━━━━━ 0s 195ms/step - accuracy: 0.9925 - loss: 0.0227
Epoch 20: val_accuracy did not improve from 0.99505
278/278 ━━━━━━━━━━━━━━━━━━━━ 68s 244ms/step - accuracy: 0.9915 - loss: 0.0256 - val_accuracy: 0.9833 - val_loss: 0.0457

Training done
Model saved to: /content/drive/MyDrive/Cataract/InceptionResNetV2_FineTune.h5


In [13]:
# Do NOT use load_model — use Cataract_Model directly from training session
# InceptionResNetV2 has CustomScaleLayer inside which breaks load_model in new session
best_model = Cataract_Model
print("Model ready")
print("Input shape:", best_model.input_shape)
print("Output shape:", best_model.output_shape)

Model ready
Input shape: (None, 224, 224, 3)
Output shape: (None, 3)


In [ ]:
test_it.reset()
test_loss, test_accuracy = best_model.evaluate(test_it)
print("Test Loss:    ", round(test_loss, 4))
print("Test Accuracy:", round(test_accuracy, 4))

80/80 ━━━━━━━━━━━━━━━━━━━━ 895s 11s/step - accuracy: 0.9808 - loss: 0.0570
Test Loss:     0.057
Test Accuracy: 0.9808


In [ ]:
test_it.reset()
y_pred_probs = best_model.predict(test_it)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_it.classes

recall = recall_score(y_true, y_pred, average='weighted')
f2     = fbeta_score(y_true, y_pred, beta=2, average='weighted')

print("Recall:  ", round(recall, 4))
print("F2 Score:", round(f2, 4))

print("\nClassification Report:")
print(classification_report(y_true, y_pred,
      target_names=list(test_it.class_indices.keys())))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

80/80 ━━━━━━━━━━━━━━━━━━━━ 41s 367ms/step
Recall:   0.9851
F2 Score: 0.985

Classification Report:
              precision    recall  f1-score   support

    Cataract       1.00      0.95      0.98       800
      Normal       0.96      1.00      0.98       800
     Not Eye       1.00      1.00      1.00       952

    accuracy                           0.99      2552
   macro avg       0.98      0.98      0.98      2552
weighted avg       0.99      0.99      0.99      2552

Confusion Matrix:
[[763  37   0]
 [  1 799   0]
 [  0   0 952]]


In [15]:
import time
import numpy as np
import os

test_it.reset()
single_image_input = next(iter(test_it))[0][:1]
print(f"Input shape: {single_image_input.shape}  ← batch_size=1 (single image)")

print("Running warm-up inferences...")
for _ in range(10):
    best_model.predict(single_image_input, verbose=0)

N = 100
times = []
for _ in range(N):
    start = time.perf_counter()
    best_model.predict(single_image_input, verbose=0)
    end   = time.perf_counter()
    times.append((end - start) * 1000)

# ── Save 100 values to Drive ──────────────────────────────
np.save("/content/drive/MyDrive/Cataract/latency_InceptionResNetV2.npy", np.array(times))
print("✅ Saved latency_InceptionResNetV2.npy")

latency_mean = np.mean(times)
latency_std  = np.std(times)
latency_p95  = np.percentile(times, 95)
model_size_mb = 214.79  # MB hardcoded

print("\n" + "="*55)
print("  InceptionResNetV2 — Single-Image Latency Report")
print("="*55)
print(f"  Average Latency : {latency_mean:.2f} ± {latency_std:.2f} ms")
print(f"  P95 Latency     : {latency_p95:.2f} ms")
print(f"  Model File Size : {model_size_mb:.2f} MB")
print(f"  Benchmark Runs  : {N}")
print(f"  Input Shape     : {single_image_input.shape}")
print(f"  Hardware        : Google Colab T4 GPU")
print("="*55)

Input shape: (1, 224, 224, 3)  ← batch_size=1 (single image)
Running warm-up inferences...
✅ Saved latency_InceptionResNetV2.npy

  InceptionResNetV2 — Single-Image Latency Report
  Average Latency : 103.30 ± 20.68 ms
  P95 Latency     : 130.08 ms
  Model File Size : 214.79 MB
  Benchmark Runs  : 100
  Input Shape     : (1, 224, 224, 3)
  Hardware        : Google Colab T4 GPU
